# 41. The Terminal Concession Bidding Problem

## Tier 4: Supervised Learning for Win Probability Prediction

### Goal
Implement machine learning enhancement for concession bidding that transforms intuition-based decisions to data-driven strategy optimization through supervised learning models that predict win probabilities and optimize bid components.

### Key assumptions
- Historical bidding data contains patterns that can predict outcomes
- Multiple ML algorithms capture different aspects of the bidding process
- Feature engineering can extract meaningful predictive signals
- Ensemble methods improve prediction accuracy and robustness

### Approach (step-by-step)
1. Create synthetic historical bidding dataset with realistic features
2. Engineer comprehensive features for win probability prediction
3. Implement ensemble of ML algorithms (Random Forest, Gradient Boosting, SVM)
4. Perform feature importance analysis and model interpretation
5. Optimize bidding strategy based on ML predictions
6. Validate model performance and compare with traditional approaches

### What to look for in the results
- Feature importance rankings and predictive insights
- Model performance metrics (accuracy, precision, recall, F1-score)
- Win probability predictions for different bidding strategies
- ML-optimized bidding recommendations
- Comparison with heuristic and optimization approaches

### Concrete example (from the source)
ML-enhanced concession bidding with:
- 200 historical bids with 45% win rate for training
- 15 engineered features including financial, competitive, and market metrics
- Ensemble methods: Random Forest, Gradient Boosting, Support Vector Machines
- Expected win probability improvement: 74.2% vs traditional approaches
- Key insights: Financial features (35% importance), Competitive position (25%), Technical capabilities (20%)

### Why this Tier exists vs earlier Tiers
Tier 4 leverages data-driven machine learning to discover complex patterns and relationships in historical bidding data that are not captured by mathematical models or heuristics. Unlike the deterministic approaches in Tiers 1-3, ML can learn from experience and provide probabilistic predictions that account for real-world complexities and market dynamics.

### Pros / Cons vs earlier Tiers
**Pros:**
- Learns from historical data and real-world outcomes
- Captures complex non-linear relationships and interactions
- Provides probabilistic predictions with confidence intervals
- Adapts to changing market conditions through retraining
- Offers explainable AI insights through feature importance

**Cons:**
- Requires substantial historical data for training
- Model performance depends on data quality and representativeness
- May overfit to historical patterns and miss novel strategies
- Complex to implement and interpret correctly
- Computational cost for training and hyperparameter tuning

### When to use this Tier
- When sufficient historical bidding data is available
- Complex bidding environments with many influencing factors
- Dynamic markets where patterns evolve over time
- When predictive accuracy is more important than optimality guarantees
- Organizations with data science capabilities and resources

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries for machine learning implementation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# Machine learning imports
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, classification_report, roc_curve
from sklearn.inspection import permutation_importance

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Define data structures for ML-enhanced bidding
@dataclass
class HistoricalBid:
    """Represents a historical concession bid with outcome"""
    bid_id: str
    # Financial features
    upfront_payment: float  # Million euros
    annual_revenue_commitment: float
    total_investment: float
    financial_capacity_score: float
    
    # Technical features
    automation_level: float  # 0-100
    technical_experience_score: float
    digital_integration_score: float
    innovation_score: float
    
    # Competitive features
    market_share: float  # Percentage
    number_of_competitors: int
    competitive_position: float  # 1-5 ranking
    regional_presence: float  # 0-100
    
    # Risk features
    political_risk_score: float  # 0-100 (higher = more risk)
    regulatory_complexity: float  # 0-100
    economic_stability: float  # 0-100
    currency_risk: float  # 0-100
    
    # Strategic features
    network_synergy_score: float  # 0-100
    customer_overlap: float  # 0-100
    strategic_alignment: float  # 0-100
    
    # Outcome
    won_concession: bool  # Target variable
    actual_win_probability: float  # If available

@dataclass
class MLModelResults:
    """Results from ML model training and evaluation"""
    model_name: str
    accuracy: float
    precision: float
    recall: float
    f1_score: float
    auc_roc: float
    feature_importance: Dict[str, float]
    confusion_matrix: np.ndarray
    training_time: float

@dataclass
class BiddingStrategyML:
    """ML-optimized bidding strategy"""
    strategy_name: str
    predicted_win_probability: float
    confidence_interval: Tuple[float, float]
    recommended_bids: Dict[str, float]
    key_drivers: List[Tuple[str, float]]  # Feature, importance
    risk_assessment: str

In [ ]:
try:
    # Generate synthetic historical bidding dataset
    def generate_historical_bidding_data(n_samples: int = 200) -> List[HistoricalBid]:
        """Generate realistic synthetic historical bidding data"""
        
        historical_bids = []
        np.random.seed(42)  # For reproducibility
        
        for i in range(n_samples):
            # Generate correlated features to simulate realistic bidding scenarios
            
            # Financial features (correlated with success)
            financial_strength = np.random.normal(0.5, 0.2)  # Base financial strength
            upfront_payment = 60 + financial_strength * 40 + np.random.normal(0, 10)
            upfront_payment = np.clip(upfront_payment, 40, 120)
            
            annual_revenue_commitment = 1.0 + financial_strength * 0.8 + np.random.normal(0, 0.2)
            annual_revenue_commitment = np.clip(annual_revenue_commitment, 0.5, 2.5)
            
            total_investment = upfront_payment + np.random.uniform(20, 80)
            financial_capacity_score = 60 + financial_strength * 30 + np.random.normal(0, 10)
            financial_capacity_score = np.clip(financial_capacity_score, 30, 100)
            
            # Technical features
            tech_strength = np.random.normal(0.6, 0.15)
            automation_level = 40 + tech_strength * 40 + np.random.normal(0, 10)
            automation_level = np.clip(automation_level, 20, 95)
            
            technical_experience_score = 70 + tech_strength * 20 + np.random.normal(0, 8)
            technical_experience_score = np.clip(technical_experience_score, 40, 100)
            
            digital_integration_score = 50 + tech_strength * 35 + np.random.normal(0, 10)
            digital_integration_score = np.clip(digital_integration_score, 25, 95)
            
            innovation_score = 55 + tech_strength * 30 + np.random.normal(0, 10)
            innovation_score = np.clip(innovation_score, 30, 100)
            
            # Competitive features
            competitive_strength = np.random.normal(0.4, 0.2)
            market_share = 5 + competitive_strength * 20 + np.random.normal(0, 5)
            market_share = np.clip(market_share, 1, 30)
            
            number_of_competitors = np.random.poisson(4) + 2  # 2-10 competitors typical
            competitive_position = max(1, min(5, int(6 - competitive_strength * 4 + np.random.normal(0, 1))))
            regional_presence = 30 + competitive_strength * 50 + np.random.normal(0, 10)
            regional_presence = np.clip(regional_presence, 10, 95)
            
            # Risk features
            risk_environment = np.random.normal(0.5, 0.2)
            political_risk_score = 20 + risk_environment * 60 + np.random.normal(0, 10)
            political_risk_score = np.clip(political_risk_score, 5, 90)
            
            regulatory_complexity = 30 + (1-risk_environment) * 40 + np.random.normal(0, 10)
            regulatory_complexity = np.clip(regulatory_complexity, 10, 85)
            
            economic_stability = 80 - risk_environment * 30 + np.random.normal(0, 10)
            economic_stability = np.clip(economic_stability, 30, 100)
            
            currency_risk = 15 + risk_environment * 25 + np.random.normal(0, 5)
            currency_risk = np.clip(currency_risk, 5, 70)
            
            # Strategic features
            strategic_strength = np.random.normal(0.5, 0.15)
            network_synergy_score = 40 + strategic_strength * 40 + np.random.normal(0, 10)
            network_synergy_score = np.clip(network_synergy_score, 15, 90)
            
            customer_overlap = 30 + strategic_strength * 35 + np.random.normal(0, 10)
            customer_overlap = np.clip(customer_overlap, 10, 85)
            
            strategic_alignment = 45 + strategic_strength * 35 + np.random.normal(0, 10)
            strategic_alignment = np.clip(strategic_alignment, 20, 95)
            
            # Calculate win probability based on weighted combination of factors
            win_probability = (
                0.25 * financial_strength +  # Financial factors
                0.20 * tech_strength +      # Technical factors
                0.20 * competitive_strength + # Competitive factors
                0.15 * (1 - risk_environment) + # Low risk helps
                0.10 * strategic_strength +   # Strategic factors
                0.10 * np.random.normal(0, 0.1)  # Random noise
            )
            
            win_probability = np.clip(win_probability, 0.1, 0.95)
            
            # Determine outcome based on probability
            won_concession = np.random.random() < win_probability
            
            # Create historical bid
            bid = HistoricalBid(
                bid_id=f"BID_{i+1:04d}",
                upfront_payment=upfront_payment,
                annual_revenue_commitment=annual_revenue_commitment,
                total_investment=total_investment,
                financial_capacity_score=financial_capacity_score,
                automation_level=automation_level,
                technical_experience_score=technical_experience_score,
                digital_integration_score=digital_integration_score,
                innovation_score=innovation_score,
                market_share=market_share,
                number_of_competitors=number_of_competitors,
                competitive_position=competitive_position,
                regional_presence=regional_presence,
                political_risk_score=political_risk_score,
                regulatory_complexity=regulatory_complexity,
                economic_stability=economic_stability,
                currency_risk=currency_risk,
                network_synergy_score=network_synergy_score,
                customer_overlap=customer_overlap,
                strategic_alignment=strategic_alignment,
                won_concession=won_concession,
                actual_win_probability=win_probability
            )
            
            historical_bids.append(bid)
        
        return historical_bids
    
    # Generate the dataset
    historical_data = generate_historical_bidding_data(200)
    print(f"Generated {len(historical_data)} historical bidding records")
    print(f"Win rate: {sum(1 for bid in historical_data if bid.won_concession) / len(historical_data):.1%}")
except Exception as e:
    print(f'Error in cell: {e}')

[Timeout after 120s - cell wrapped for safety]

In [ ]:
try:
    # Convert to DataFrame and perform feature engineering
    def prepare_ml_dataset(historical_bids: List[HistoricalBid]) -> pd.DataFrame:
        """Convert historical bids to ML-ready DataFrame with engineered features"""
        
        # Convert to DataFrame
        data = []
        for bid in historical_bids:
            row = {
                'upfront_payment': bid.upfront_payment,
                'annual_revenue_commitment': bid.annual_revenue_commitment,
                'total_investment': bid.total_investment,
                'financial_capacity_score': bid.financial_capacity_score,
                'automation_level': bid.automation_level,
                'technical_experience_score': bid.technical_experience_score,
                'digital_integration_score': bid.digital_integration_score,
                'innovation_score': bid.innovation_score,
                'market_share': bid.market_share,
                'number_of_competitors': bid.number_of_competitors,
                'competitive_position': bid.competitive_position,
                'regional_presence': bid.regional_presence,
                'political_risk_score': bid.political_risk_score,
                'regulatory_complexity': bid.regulatory_complexity,
                'economic_stability': bid.economic_stability,
                'currency_risk': bid.currency_risk,
                'network_synergy_score': bid.network_synergy_score,
                'customer_overlap': bid.customer_overlap,
                'strategic_alignment': bid.strategic_alignment,
                'won_concession': int(bid.won_concession),  # Convert to int for ML
                'actual_win_probability': bid.actual_win_probability
            }
            data.append(row)
        
        df = pd.DataFrame(data)
        
        # Feature engineering
        
        # 1. Financial ratios and composites
        df['payment_to_investment_ratio'] = df['upfront_payment'] / df['total_investment']
        df['financial_strength_index'] = (
            df['financial_capacity_score'] * 0.4 +
            (df['upfront_payment'] / df['upfront_payment'].max()) * 100 * 0.3 +
            (df['annual_revenue_commitment'] / df['annual_revenue_commitment'].max()) * 100 * 0.3
        )
        
        # 2. Technical capability composite
        df['technical_capability_score'] = (
            df['automation_level'] * 0.3 +
            df['technical_experience_score'] * 0.3 +
            df['digital_integration_score'] * 0.2 +
            df['innovation_score'] * 0.2
        )
        
        # 3. Competitive intensity metrics
        df['competitive_intensity'] = df['number_of_competitors'] / df['competitive_position']
        df['market_dominance_score'] = df['market_share'] * df['regional_presence'] / 100
        
        # 4. Risk assessment composites
        df['overall_risk_score'] = (
            df['political_risk_score'] * 0.3 +
            df['regulatory_complexity'] * 0.25 +
            (100 - df['economic_stability']) * 0.25 +  # Invert stability
            df['currency_risk'] * 0.2
        )
        
        df['risk_adjusted_potential'] = df['technical_capability_score'] / (1 + df['overall_risk_score']/100)
        
        # 5. Strategic alignment composite
        df['strategic_fit_score'] = (
            df['network_synergy_score'] * 0.4 +
            df['customer_overlap'] * 0.3 +
            df['strategic_alignment'] * 0.3
        )
        
        # 6. Interaction terms
        df['tech_financial_synergy'] = df['technical_capability_score'] * df['financial_strength_index'] / 100
        df['competitive_risk_interaction'] = df['market_dominance_score'] / (1 + df['overall_risk_score']/100)
        
        return df
    
    # Prepare the ML dataset
    ml_df = prepare_ml_dataset(historical_data)
    print(f"ML dataset shape: {ml_df.shape}")
    print(f"Engineered features: {ml_df.shape[1] - 2}")  # -2 for target and actual probability
    print(f"\nFeature summary:")
    print(ml_df.describe().round(2))
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'historical_data' is not defined...]

In [ ]:
# Machine Learning Pipeline Implementation
class ConcessionBiddingML:
    """Machine Learning pipeline for concession bidding optimization"""
    
    def __init__(self):
        self.models = {}
        self.scalers = {}
        self.feature_names = []
        self.model_results = {}
        
    def prepare_data(self, df: pd.DataFrame, test_size: float = 0.2) -> Tuple:
        """Prepare data for ML training"""
        
        # Separate features and target
        feature_cols = [col for col in df.columns if col not in ['won_concession', 'actual_win_probability']]
        self.feature_names = feature_cols
        
        X = df[feature_cols]
        y = df['won_concession']
        
        # Split data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42, stratify=y
        )
        
        # Scale features for algorithms that need it
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        self.scalers['standard'] = scaler
        
        return X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test
    
    def train_random_forest(self, X_train, y_train) -> MLModelResults:
        """Train Random Forest model"""
        
        start_time = time.time()
        
        # Hyperparameter tuning
        param_grid = {
            'n_estimators': [100, 200],
            'max_depth': [10, 15, None],
            'min_samples_split': [2, 5],
            'min_samples_leaf': [1, 2]
        }
        
        rf = RandomForestClassifier(random_state=42)
        grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='f1', n_jobs=-1)
        grid_search.fit(X_train, y_train)
        
        best_model = grid_search.best_estimator_
        training_time = time.time() - start_time
        
        # Extract feature importance
        feature_importance = dict(zip(self.feature_names, best_model.feature_importances_))
        
        return MLModelResults(
            model_name="Random Forest",
            accuracy=0,  # Will be calculated during evaluation
            precision=0,
            recall=0,
            f1_score=0,
            auc_roc=0,
            feature_importance=feature_importance,
            confusion_matrix=np.array([]),
            training_time=training_time
        ), best_model
    
    def train_gradient_boosting(self, X_train, y_train) -> MLModelResults:
        """Train Gradient Boosting model"""
        
        start_time = time.time()
        
        # Hyperparameter tuning
        param_grid = {
            'n_estimators': [100, 200],
            'learning_rate': [0.05, 0.1],
            'max_depth': [3, 5],
            'subsample': [0.8, 1.0]
        }
        
        gb = GradientBoostingClassifier(random_state=42)
        grid_search = GridSearchCV(gb, param_grid, cv=5, scoring='f1', n_jobs=-1)
        grid_search.fit(X_train, y_train)
        
        best_model = grid_search.best_estimator_
        training_time = time.time() - start_time
        
        # Extract feature importance
        feature_importance = dict(zip(self.feature_names, best_model.feature_importances_))
        
        return MLModelResults(
            model_name="Gradient Boosting",
            accuracy=0,
            precision=0,
            recall=0,
            f1_score=0,
            auc_roc=0,
            feature_importance=feature_importance,
            confusion_matrix=np.array([]),
            training_time=training_time
        ), best_model
    
    def train_svm(self, X_train_scaled, y_train) -> MLModelResults:
        """Train Support Vector Machine model"""
        
        start_time = time.time()
        
        # Hyperparameter tuning
        param_grid = {
            'C': [1, 10, 100],
            'gamma': ['scale', 'auto'],
            'kernel': ['rbf']
        }
        
        svm = SVC(probability=True, random_state=42)
        grid_search = GridSearchCV(svm, param_grid, cv=5, scoring='f1', n_jobs=-1)
        grid_search.fit(X_train_scaled, y_train)
        
        best_model = grid_search.best_estimator_
        training_time = time.time() - start_time
        
        # SVM doesn't have built-in feature importance, use permutation importance
        # For now, use placeholder (will be calculated later)
        feature_importance = {name: 0.0 for name in self.feature_names}
        
        return MLModelResults(
            model_name="Support Vector Machine",
            accuracy=0,
            precision=0,
            recall=0,
            f1_score=0,
            auc_roc=0,
            feature_importance=feature_importance,
            confusion_matrix=np.array([]),
            training_time=training_time
        ), best_model
    
    def evaluate_model(self, model, X_test, y_test, model_name: str, scaled: bool = False) -> MLModelResults:
        """Evaluate trained model"""
        
        # Make predictions
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
        
        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average='binary')
        recall = recall_score(y_test, y_pred, average='binary')
        f1 = f1_score(y_test, y_pred, average='binary')
        auc_roc = roc_auc_score(y_test, y_pred_proba) if y_pred_proba is not None else 0
        
        # Confusion matrix
        cm = confusion_matrix(y_test, y_pred)
        
        # Update results
        if model_name in self.model_results:
            results = self.model_results[model_name]
            results.accuracy = accuracy
            results.precision = precision
            results.recall = recall
            results.f1_score = f1
            results.auc_roc = auc_roc
            results.confusion_matrix = cm
            
            return results
        
        return MLModelResults(
            model_name=model_name,
            accuracy=accuracy,
            precision=precision,
            recall=recall,
            f1_score=f1,
            auc_roc=auc_roc,
            feature_importance={},
            confusion_matrix=cm,
            training_time=0
        )
    
    def train_all_models(self, df: pd.DataFrame) -> Dict[str, MLModelResults]:
        """Train all ML models"""
        
        print("=== Training Machine Learning Models ===")
        
        # Prepare data
        X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test = self.prepare_data(df)
        
        print(f"Training set size: {X_train.shape[0]}")
        print(f"Test set size: {X_test.shape[0]}")
        print(f"Number of features: {X_train.shape[1]}")
        
        # Train Random Forest
        print("\nTraining Random Forest...")
        rf_results, rf_model = self.train_random_forest(X_train, y_train)
        rf_evaluated = self.evaluate_model(rf_model, X_test, y_test, "Random Forest")
        self.models['random_forest'] = rf_model
        self.model_results['random_forest'] = rf_evaluated
        
        # Train Gradient Boosting
        print("Training Gradient Boosting...")
        gb_results, gb_model = self.train_gradient_boosting(X_train, y_train)
        gb_evaluated = self.evaluate_model(gb_model, X_test, y_test, "Gradient Boosting")
        self.models['gradient_boosting'] = gb_model
        self.model_results['gradient_boosting'] = gb_evaluated
        
        # Train SVM
        print("Training Support Vector Machine...")
        svm_results, svm_model = self.train_svm(X_train_scaled, y_train)
        svm_evaluated = self.evaluate_model(svm_model, X_test_scaled, y_test, "Support Vector Machine", scaled=True)
        self.models['svm'] = svm_model
        self.model_results['svm'] = svm_evaluated
        
        return self.model_results
    
    def predict_win_probability(self, bid_features: Dict[str, float]) -> Dict[str, float]:
        """Predict win probability for a new bid using all models"""
        
        # Convert to DataFrame
        feature_df = pd.DataFrame([bid_features])
        
        # Ensure all required features are present
        for feature in self.feature_names:
            if feature not in feature_df.columns:
                feature_df[feature] = 0  # Default value
        
        feature_df = feature_df[self.feature_names]
        
        predictions = {}
        
        # Random Forest prediction
        if 'random_forest' in self.models:
            rf_pred = self.models['random_forest'].predict_proba(feature_df)[0, 1]
            predictions['random_forest'] = rf_pred
        
        # Gradient Boosting prediction
        if 'gradient_boosting' in self.models:
            gb_pred = self.models['gradient_boosting'].predict_proba(feature_df)[0, 1]
            predictions['gradient_boosting'] = gb_pred
        
        # SVM prediction
        if 'svm' in self.models:
            feature_df_scaled = self.scalers['standard'].transform(feature_df)
            svm_pred = self.models['svm'].predict_proba(feature_df_scaled)[0, 1]
            predictions['svm'] = svm_pred
        
        # Ensemble prediction (average)
        if predictions:
            predictions['ensemble'] = np.mean(list(predictions.values()))
        
        return predictions

In [ ]:
try:
    # Train the ML models
    ml_pipeline = ConcessionBiddingML()
    model_results = ml_pipeline.train_all_models(ml_df)
    
    # Display model performance
    print("\n=== Model Performance Summary ===")
    performance_df = pd.DataFrame([
        {
            'Model': results.model_name,
            'Accuracy': results.accuracy,
            'Precision': results.precision,
            'Recall': results.recall,
            'F1-Score': results.f1_score,
            'AUC-ROC': results.auc_roc,
            'Training Time (s)': results.training_time
        }
        for results in model_results.values()
    ])
    
    print(performance_df.round(4).to_string(index=False))
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'ml_df' is not defined...]

In [ ]:
try:
    # Feature importance analysis
    def analyze_feature_importance(model_results: Dict[str, MLModelResults]):
        """Analyze and visualize feature importance across models"""
        
        print("\n=== Feature Importance Analysis ===")
        
        # Combine feature importance from tree-based models
        combined_importance = {}
        
        for model_name, results in model_results.items():
            if results.feature_importance:
                print(f"\n{model_name} - Top 10 Features:")
                
                # Sort features by importance
                sorted_features = sorted(results.feature_importance.items(), 
                                         key=lambda x: x[1], reverse=True)[:10]
                
                for feature, importance in sorted_features:
                    print(f"  {feature}: {importance:.4f}")
                    
                    # Combine importance (average across models)
                    if feature not in combined_importance:
                        combined_importance[feature] = []
                    combined_importance[feature].append(importance)
        
        # Calculate average importance
        avg_importance = {}
        for feature, values in combined_importance.items():
            avg_importance[feature] = np.mean(values)
        
        # Sort by average importance
        sorted_avg_importance = sorted(avg_importance.items(), key=lambda x: x[1], reverse=True)
        
        print(f"\nCombined Feature Importance (Top 15):")
        for feature, importance in sorted_avg_importance[:15]:
            print(f"  {feature}: {importance:.4f}")
        
        return sorted_avg_importance
    
    # Analyze feature importance
    feature_importance = analyze_feature_importance(model_results)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'model_results' is not defined...]

In [ ]:
try:
    # Create comprehensive visualization of ML results
    def visualize_ml_results(model_results: Dict[str, MLModelResults], feature_importance: List[Tuple[str, float]]):
        """Create comprehensive visualization of ML analysis results"""
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle('Machine Learning Analysis - Terminal Concession Bidding', fontsize=16, fontweight='bold')
        
        # 1. Model Performance Comparison
        ax1 = axes[0, 0]
        models = list(model_results.keys())
        metrics = ['accuracy', 'precision', 'recall', 'f1_score', 'auc_roc']
        
        x_pos = np.arange(len(models))
        width = 0.15
        
        for i, metric in enumerate(metrics):
            values = [getattr(model_results[model], metric) for model in models]
            ax1.bar(x_pos + i * width, values, width, label=metric.replace('_', ' ').title(),
                   alpha=0.8)
        
        ax1.set_xlabel('Machine Learning Models')
        ax1.set_ylabel('Performance Score')
        ax1.set_title('Model Performance Comparison')
        ax1.set_xticks(x_pos + width * 2)
        ax1.set_xticklabels([model_results[model].model_name for model in models], rotation=45)
        ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax1.grid(True, alpha=0.3)
        ax1.set_ylim(0, 1)
        
        # 2. Feature Importance (Top 10)
        ax2 = axes[0, 1]
        top_features = feature_importance[:10]
        features, importance = zip(*top_features)
        
        # Truncate long feature names for display
        display_features = [f[:20] + '...' if len(f) > 20 else f for f in features]
        
        bars2 = ax2.barh(range(len(features)), importance, color='skyblue', alpha=0.8)
        ax2.set_yticks(range(len(features)))
        ax2.set_yticklabels(display_features)
        ax2.set_xlabel('Average Importance Score')
        ax2.set_title('Top 10 Feature Importance')
        ax2.grid(True, alpha=0.3)
        
        # Add importance values on bars
        for i, (bar, imp) in enumerate(zip(bars2, importance)):
            ax2.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
                    f'{imp:.3f}', ha='left', va='center', fontweight='bold')
        
        # 3. Confusion Matrices
        ax3 = axes[0, 2]
        
        # Create combined confusion matrix visualization
        models_list = list(model_results.keys())
        cms = [model_results[model].confusion_matrix for model in models_list]
        
        # Sum confusion matrices for overall performance
        combined_cm = sum(cms)
        
        im = ax3.imshow(combined_cm, interpolation='nearest', cmap=plt.cm.Blues)
        ax3.figure.colorbar(im, ax=ax3)
        
        # Add text annotations
        thresh = combined_cm.max() / 2.
        for i in range(combined_cm.shape[0]):
            for j in range(combined_cm.shape[1]):
                ax3.text(j, i, format(combined_cm[i, j], 'd'),
                        ha="center", va="center",
                        color="white" if combined_cm[i, j] > thresh else "black")
        
        ax3.set_xlabel('Predicted Label')
        ax3.set_ylabel('True Label')
        ax3.set_title('Combined Confusion Matrix')
        ax3.set_xticks([0, 1])
        ax3.set_yticks([0, 1])
        ax3.set_xticklabels(['Lost', 'Won'])
        ax3.set_yticklabels(['Lost', 'Won'])
        
        # 4. Feature Categories Analysis
        ax4 = axes[1, 0]
        
        # Group features by categories
        category_importance = {
            'Financial': 0,
            'Technical': 0,
            'Competitive': 0,
            'Risk': 0,
            'Strategic': 0,
            'Interaction': 0
        }
        
        category_keywords = {
            'Financial': ['payment', 'investment', 'financial', 'revenue'],
            'Technical': ['automation', 'technical', 'digital', 'innovation'],
            'Competitive': ['market', 'competitive', 'competitors'],
            'Risk': ['risk', 'political', 'regulatory', 'economic', 'currency'],
            'Strategic': ['strategic', 'synergy', 'alignment', 'customer'],
            'Interaction': ['synergy', 'interaction']
        }
        
        for feature, importance in feature_importance:
            feature_lower = feature.lower()
            for category, keywords in category_keywords.items():
                if any(keyword in feature_lower for keyword in keywords):
                    category_importance[category] += importance
                    break
        
        # Create pie chart
        categories = list(category_importance.keys())
        values = list(category_importance.values())
        colors4 = plt.cm.Set3(np.linspace(0, 1, len(categories)))
        
        wedges, texts, autotexts = ax4.pie(values, labels=categories, autopct='%1.1f%%', 
                                            colors=colors4, startangle=90)
        ax4.set_title('Feature Importance by Category')
        
        # 5. Training Time vs Performance
        ax5 = axes[1, 1]
        
        training_times = [model_results[model].training_time for model in models_list]
        f1_scores = [model_results[model].f1_score for model in models_list]
        model_names = [model_results[model].model_name for model in models_list]
        
        scatter = ax5.scatter(training_times, f1_scores, s=200, alpha=0.7, c=range(len(models)), cmap='viridis')
        
        # Add model labels
        for i, (x, y, name) in enumerate(zip(training_times, f1_scores, model_names)):
            ax5.annotate(name, (x, y), xytext=(5, 5), textcoords='offset points', fontsize=9)
        
        ax5.set_xlabel('Training Time (seconds)')
        ax5.set_ylabel('F1-Score')
        ax5.set_title('Training Efficiency vs Performance')
        ax5.grid(True, alpha=0.3)
        
        # 6. Prediction Confidence Distribution
        ax6 = axes[1, 2]
        
        # Generate prediction confidence for test set
        X_train, X_test, X_train_scaled, X_test_scaled, y_train, y_test = ml_pipeline.prepare_data(ml_df)
        
        # Get prediction probabilities from best model (usually Gradient Boosting or Random Forest)
        best_model_name = max(model_results.keys(), 
                              key=lambda x: getattr(model_results[x], 'f1_score'))
        
        if best_model_name in ['random_forest', 'gradient_boosting']:
            model = ml_pipeline.models[best_model_name]
            y_pred_proba = model.predict_proba(X_test)[:, 1]
            
            # Separate predictions by actual outcome
            won_probs = y_pred_proba[y_test == 1]
            lost_probs = y_pred_proba[y_test == 0]
            
            ax6.hist(lost_probs, bins=15, alpha=0.7, label='Lost Concession', color='red', density=True)
            ax6.hist(won_probs, bins=15, alpha=0.7, label='Won Concession', color='green', density=True)
            ax6.set_xlabel('Predicted Win Probability')
            ax6.set_ylabel('Density')
            ax6.set_title(f'Prediction Confidence Distribution ({model_results[best_model_name].model_name})')
            ax6.legend()
            ax6.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        return fig
    
    # Visualize the results
    fig = visualize_ml_results(model_results, feature_importance)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'model_results' is not defined...]

In [ ]:
try:
    # ML-optimized bidding strategy generation
    def generate_ml_optimized_strategy(ml_pipeline: ConcessionBiddingML, 
                                     model_results: Dict[str, MLModelResults]) -> BiddingStrategyML:
        """Generate ML-optimized bidding strategy"""
        
        print("\n=== ML-Optimized Bidding Strategy Generation ===")
        
        # Find best performing model
        best_model_name = max(model_results.keys(), 
                              key=lambda x: getattr(model_results[x], 'f1_score'))
        best_model = ml_pipeline.models[best_model_name]
        
        print(f"Using best model: {model_results[best_model_name].model_name}")
        print(f"Model F1-Score: {model_results[best_model_name].f1_score:.4f}")
        
        # Generate candidate bidding scenarios
        scenarios = []
        
        # Scenario 1: Conservative approach
        conservative_features = {
            'upfront_payment': 75.0,
            'annual_revenue_commitment': 1.4,
            'total_investment': 120.0,
            'financial_capacity_score': 80.0,
            'automation_level': 60.0,
            'technical_experience_score': 75.0,
            'digital_integration_score': 65.0,
            'innovation_score': 70.0,
            'market_share': 12.0,
            'number_of_competitors': 4,
            'competitive_position': 2,
            'regional_presence': 60.0,
            'political_risk_score': 30.0,
            'regulatory_complexity': 40.0,
            'economic_stability': 80.0,
            'currency_risk': 20.0,
            'network_synergy_score': 60.0,
            'customer_overlap': 50.0,
            'strategic_alignment': 65.0,
        }
        
        # Calculate engineered features for conservative scenario
        conservative_features.update({
            'payment_to_investment_ratio': conservative_features['upfront_payment'] / conservative_features['total_investment'],
            'financial_strength_index': (
                conservative_features['financial_capacity_score'] * 0.4 +
                (conservative_features['upfront_payment'] / 120) * 100 * 0.3 +
                (conservative_features['annual_revenue_commitment'] / 2.5) * 100 * 0.3
            ),
            'technical_capability_score': (
                conservative_features['automation_level'] * 0.3 +
                conservative_features['technical_experience_score'] * 0.3 +
                conservative_features['digital_integration_score'] * 0.2 +
                conservative_features['innovation_score'] * 0.2
            ),
            'competitive_intensity': conservative_features['number_of_competitors'] / conservative_features['competitive_position'],
            'market_dominance_score': conservative_features['market_share'] * conservative_features['regional_presence'] / 100,
            'overall_risk_score': (
                conservative_features['political_risk_score'] * 0.3 +
                conservative_features['regulatory_complexity'] * 0.25 +
                (100 - conservative_features['economic_stability']) * 0.25 +
                conservative_features['currency_risk'] * 0.2
            ),
            'risk_adjusted_potential': 0,  # Will be calculated below
            'strategic_fit_score': (
                conservative_features['network_synergy_score'] * 0.4 +
                conservative_features['customer_overlap'] * 0.3 +
                conservative_features['strategic_alignment'] * 0.3
            ),
            'tech_financial_synergy': 0,  # Will be calculated below
            'competitive_risk_interaction': 0  # Will be calculated below
        })
        
        # Calculate remaining engineered features
        conservative_features['risk_adjusted_potential'] = (
            conservative_features['technical_capability_score'] / 
            (1 + conservative_features['overall_risk_score']/100)
        )
        conservative_features['tech_financial_synergy'] = (
            conservative_features['technical_capability_score'] * 
            conservative_features['financial_strength_index'] / 100
        )
        conservative_features['competitive_risk_interaction'] = (
            conservative_features['market_dominance_score'] / 
            (1 + conservative_features['overall_risk_score']/100)
        )
        
        # Get prediction for conservative scenario
        conservative_predictions = ml_pipeline.predict_win_probability(conservative_features)
        
        # Scenario 2: Aggressive approach (higher investment, better technology)
        aggressive_features = conservative_features.copy()
        aggressive_features.update({
            'upfront_payment': 95.0,
            'annual_revenue_commitment': 1.8,
            'total_investment': 160.0,
            'financial_capacity_score': 90.0,
            'automation_level': 85.0,
            'technical_experience_score': 90.0,
            'digital_integration_score': 80.0,
            'innovation_score': 85.0,
        })
        
        # Recalculate engineered features for aggressive scenario
        aggressive_features['payment_to_investment_ratio'] = aggressive_features['upfront_payment'] / aggressive_features['total_investment']
        aggressive_features['financial_strength_index'] = (
            aggressive_features['financial_capacity_score'] * 0.4 +
            (aggressive_features['upfront_payment'] / 120) * 100 * 0.3 +
            (aggressive_features['annual_revenue_commitment'] / 2.5) * 100 * 0.3
        )
        aggressive_features['technical_capability_score'] = (
            aggressive_features['automation_level'] * 0.3 +
            aggressive_features['technical_experience_score'] * 0.3 +
            aggressive_features['digital_integration_score'] * 0.2 +
            aggressive_features['innovation_score'] * 0.2
        )
        aggressive_features['risk_adjusted_potential'] = (
            aggressive_features['technical_capability_score'] / 
            (1 + aggressive_features['overall_risk_score']/100)
        )
        aggressive_features['tech_financial_synergy'] = (
            aggressive_features['technical_capability_score'] * 
            aggressive_features['financial_strength_index'] / 100
        )
        
        # Get prediction for aggressive scenario
        aggressive_predictions = ml_pipeline.predict_win_probability(aggressive_features)
        
        # Choose best scenario
        if aggressive_predictions.get('ensemble', 0) > conservative_predictions.get('ensemble', 0):
            best_features = aggressive_features
            best_predictions = aggressive_predictions
            strategy_name = "Aggressive ML-Optimized Strategy"
        else:
            best_features = conservative_features
            best_predictions = conservative_predictions
            strategy_name = "Conservative ML-Optimized Strategy"
        
        # Extract key drivers (top 5 features)
        key_drivers = feature_importance[:5]
        
        # Risk assessment
        if best_features['overall_risk_score'] > 50:
            risk_assessment = "High Risk - Requires careful mitigation strategies"
        elif best_features['overall_risk_score'] > 30:
            risk_assessment = "Medium Risk - Monitor and manage key risk factors"
        else:
            risk_assessment = "Low Risk - Favorable conditions for success"
        
        # Create strategy
        strategy = BiddingStrategyML(
            strategy_name=strategy_name,
            predicted_win_probability=best_predictions.get('ensemble', 0),
            confidence_interval=(
                max(0, best_predictions.get('ensemble', 0) - 0.1),
                min(1, best_predictions.get('ensemble', 0) + 0.1)
            ),
            recommended_bids={
                'upfront_payment': best_features['upfront_payment'],
                'annual_revenue_commitment': best_features['annual_revenue_commitment'],
                'total_investment': best_features['total_investment'],
                'automation_level': best_features['automation_level'],
                'technical_experience_score': best_features['technical_experience_score']
            },
            key_drivers=key_drivers,
            risk_assessment=risk_assessment
        )
        
        return strategy
    
    # Generate ML-optimized strategy
    ml_strategy = generate_ml_optimized_strategy(ml_pipeline, model_results)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'model_results' is not defined...]

In [ ]:
try:
    # Display ML strategy results
    def display_ml_strategy(strategy: BiddingStrategyML):
        """Display the ML-optimized bidding strategy"""
        
        print(f"\n=== ML-OPTIMIZED BIDDING STRATEGY ===")
        print(f"\nStrategy: {strategy.strategy_name}")
        print(f"Predicted Win Probability: {strategy.predicted_win_probability:.2%}")
        print(f"Confidence Interval: {strategy.confidence_interval[0]:.2%} - {strategy.confidence_interval[1]:.2%}")
        print(f"Risk Assessment: {strategy.risk_assessment}")
        
        print(f"\nRecommended Bid Components:")
        for component, value in strategy.recommended_bids.items():
            if 'payment' in component or 'investment' in component:
                print(f"- {component.replace('_', ' ').title()}: €{value:.1f} million")
            elif 'commitment' in component:
                print(f"- {component.replace('_', ' ').title()}: {value:.1f} million TEU/year")
            else:
                print(f"- {component.replace('_', ' ').title()}: {value:.1f}/100")
        
        print(f"\nKey Success Drivers (Top 5):")
        for i, (feature, importance) in enumerate(strategy.key_drivers, 1):
            print(f"{i}. {feature}: {importance:.3f}")
        
        print(f"\nML Insights:")
        print(f"- Data-driven approach based on {len(historical_data)} historical bids")
        print(f"- Ensemble prediction from {len(model_results)} ML algorithms")
        print(f"- Strategy optimized for maximum win probability")
        print(f"- Risk-adjusted recommendations for balanced approach")
    
    # Display the strategy
    display_ml_strategy(ml_strategy)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'ml_strategy' is not defined...]

In [ ]:
try:
    # Comprehensive summary and conclusions
    def generate_ml_summary(model_results: Dict[str, MLModelResults], 
                          feature_importance: List[Tuple[str, float]],
                          strategy: BiddingStrategyML):
        """Generate comprehensive summary for ML approach"""
        
        print("\n" + "="*80)
        print("TERMINAL CONCESSION BIDDING - MACHINE LEARNING ANALYSIS")
        print("="*80)
        
        print(f"\nMachine Learning Pipeline Characteristics:")
        print(f"- Approach: Supervised learning ensemble methods")
        print(f"- Algorithms: Random Forest, Gradient Boosting, Support Vector Machine")
        print(f"- Training Data: {len(historical_data)} historical bids")
        print(f"- Engineered Features: {len(ml_pipeline.feature_names)}")
        print(f"- Target Variable: Concession win/loss outcome")
        
        print(f"\nModel Performance Summary:")
        best_model = max(model_results.values(), key=lambda x: x.f1_score)
        worst_model = min(model_results.values(), key=lambda x: x.f1_score)
        
        print(f"- Best Model: {best_model.model_name} (F1: {best_model.f1_score:.4f})")
        print(f"- Worst Model: {worst_model.model_name} (F1: {worst_model.f1_score:.4f})")
        print(f"- Performance Range: {(best_model.f1_score - worst_model.f1_score):.4f}")
        print(f"- Average AUC-ROC: {np.mean([r.auc_roc for r in model_results.values()]):.4f}")
        
        print(f"\nFeature Importance Insights:")
        print(f"- Most Important Feature: {feature_importance[0][0]} ({feature_importance[0][1]:.4f})")
        print(f"- Top 3 Features: {', '.join([f[0] for f in feature_importance[:3]])}")
        
        # Categorize top features
        financial_features = [f for f in feature_importance[:10] if any(k in f[0].lower() 
                              for k in ['payment', 'investment', 'financial', 'revenue'])]
        technical_features = [f for f in feature_importance[:10] if any(k in f[0].lower() 
                             for k in ['automation', 'technical', 'digital', 'innovation'])]
        
        print(f"- Financial Features in Top 10: {len(financial_features)}")
        print(f"- Technical Features in Top 10: {len(technical_features)}")
        
        print(f"\nML-Optimized Strategy Results:")
        print(f"- Strategy Type: {strategy.strategy_name}")
        print(f"- Predicted Win Probability: {strategy.predicted_win_probability:.2%}")
        print(f"- Confidence Interval: ±{(strategy.confidence_interval[1] - strategy.confidence_interval[0])/2:.1%}")
        print(f"- Recommended Investment: €{strategy.recommended_bids['total_investment']:.1f} million")
        print(f"- Risk Level: {strategy.risk_assessment}")
        
        print(f"\nKey Predictive Insights:")
        print(f"- Financial factors: {len(financial_features)*10}% of top features")
        print(f"- Technical capabilities: {len(technical_features)*10}% of top features")
        print(f"- Competitive position: Strong predictor of success")
        print(f"- Risk factors: Significant impact on outcomes")
        
        print(f"\nML Advantages:")
        print(f"✓ Learns from historical data and real-world outcomes")
        print(f"✓ Captures complex non-linear relationships")
        print(f"✓ Provides probabilistic predictions with confidence")
        print(f"✓ Adapts to changing market conditions")
        print(f"✓ Offers explainable AI through feature importance")
        
        print(f"\nComparison with Earlier Tiers:")
        print(f"Tier 1 (MIP): Optimal but assumes perfect information")
        print(f"Tier 2 (Heuristic): Fast but limited to linear relationships")
        print(f"Tier 3 (ACO): Adaptive but requires many iterations")
        print(f"Tier 4 (ML): Data-driven with predictive capabilities")
        print(f"- Prediction Accuracy: {best_model.f1_score:.1%} (F1-Score)")
        print(f"- Adaptability: High (retrain with new data)")
        print(f"- Interpretability: Medium (feature importance analysis)")
        
        print(f"\nPractical Applications:")
        print(f"- Data-driven bid optimization and strategy development")
        print(f"- Competitive intelligence and market analysis")
        print(f"- Risk assessment and mitigation planning")
        print(f"- Portfolio optimization across multiple opportunities")
        print(f"- Continuous learning and strategy improvement")
        
        print(f"\nImplementation Requirements:")
        print(f"- Historical bidding data (200+ records recommended)")
        print(f"- Data science capabilities and resources")
        print(f"- Regular model retraining and validation")
        print(f"- Feature engineering and domain expertise")
        print(f"- Model monitoring and performance tracking")
        
        print(f"\nWhen to Use ML Approach:")
        print(f"- Sufficient historical data available (100+ bids)")
        print(f"- Complex bidding environment with many factors")
        print(f"- Dynamic markets requiring adaptive strategies")
        print(f"- Organizations with data science maturity")
        print(f"- When predictive accuracy outweighs optimality guarantees")
        
        print("\n" + "="*80)
    
    # Generate comprehensive summary
    generate_ml_summary(model_results, feature_importance, ml_strategy)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'model_results' is not defined...]